In [4]:
from pathlib import Path
import polars as pl
import tiktoken

DATA_DIR = Path("/Users/baukebrenninkmeijer/Developer/pydata-2025-context-is-king/") / "data"
(DATA_DIR / "processed").mkdir(parents=True, exist_ok=True)

encoding = tiktoken.encoding_for_model("gpt-4o")


In [ ]:
from datasets import load_dataset

ds = load_dataset("Salesforce/wikitext", "wikitext-103-v1")

README.md: 0.00B [00:00, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/722k [00:00<?, ?B/s]

train-00000-of-00002.parquet:   0%|          | 0.00/156M [00:00<?, ?B/s]

train-00001-of-00002.parquet:   0%|          | 0.00/156M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/655k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

In [2]:
import polars as pl

df = pl.scan_ndjson("~/Downloads/v1.0 simplified nq dev all.jsonl")
df.head(2).collect()

annotations,document_html,document_title,document_tokens,document_url,example_id,long_answer_candidates,question_text,question_tokens
list[struct[4]],str,str,list[struct[4]],str,i64,list[struct[5]],str,list[str]
"[{null,{92,67824,925,66429,808},[{66817,837,66588,816}],""NONE""}, {6237931520544082939,{92,67824,925,66429,808},[{66609,819,66588,816}],""NONE""}, … {5015853435362506856,{92,67824,925,66429,808},[{66609,819,66595,817}],""NONE""}]","""<!DOCTYPE html> <HTML class=""c…","""Therefore sign""","[{101,false,92,""Therefore""}, {106,false,102,""sign""}, … {103618,true,103613,""</Ul>""}]","""https://en.wikipedia.org//w/in…",6915606477668963399,"[{66428,808,41427,14,true}, {41683,20,41505,15,false}, … {75257,1540,74711,1481,true}]","""what do the 3 dots mean in mat…","[""what"", ""do"", … ""math""]"
"[{null,{-1,-1,-1,-1,-1},[],""NONE""}, {5211794810959200573,{-1,-1,-1,-1,-1},[],""NONE""}, … {null,{-1,-1,-1,-1,-1},[],""NONE""}]","""<!DOCTYPE html> <HTML class=""c…","""Watchman (law enforcement)""","[{100,false,92,""Watchman""}, {102,false,101,""(""}, … {130545,true,130540,""</Ul>""}]","""https://en.wikipedia.org//w/in…",-4505971823174084926,"[{43267,115,41595,30,true}, {42120,52,41673,32,false}, … {74963,3536,74701,3518,true}]","""when was the writ watch invent…","[""when"", ""was"", … ""who""]"


In [ ]:
counts = (
    df.select(pl.col("document_url").value_counts()).collect().unnest("document_url").sort("count", descending=True)
)
counts

document_url,count
str,u32
"""https://en.wikipedia.org//w/in…",1
"""https://en.wikipedia.org//w/in…",1
"""https://en.wikipedia.org//w/in…",1
"""https://en.wikipedia.org//w/in…",1
"""https://en.wikipedia.org//w/in…",1
…,…
"""https://en.wikipedia.org//w/in…",1
"""https://en.wikipedia.org//w/in…",1
"""https://en.wikipedia.org//w/in…",1


In [ ]:
counts.sort("count", descending=True)

document_url,count
str,u32
"""https://en.wikipedia.org//w/in…",8
"""https://en.wikipedia.org//w/in…",7
"""https://en.wikipedia.org//w/in…",6
"""https://en.wikipedia.org//w/in…",6
"""https://en.wikipedia.org//w/in…",5
…,…
"""https://en.wikipedia.org//w/in…",1
"""https://en.wikipedia.org//w/in…",1
"""https://en.wikipedia.org//w/in…",1


In [3]:
token_counts = (
    df.with_columns(
        pl.col("document_html")
        .map_elements(lambda x: len(encoding.encode(x)), return_dtype=pl.Int64)
        .alias("document_tokens")
    )
    .select("document_tokens")
    .collect()
)
token_counts

document_tokens
i64
34241
43051
82144
59866
46280
…
58626
83301
78043


In [ ]:
token_counts.select(
    pl.col("document_tokens").min().alias("min"),
    pl.col("document_tokens").max().alias("max"),
    pl.col("document_tokens").mean().alias("mean"),
)

min,max,mean
i64,i64,f64
15146,548918,76573.488378


In [ ]:
import altair as alt

alt.data_transformers.enable("vegafusion")

DataTransformerRegistry.enable('vegafusion')

In [ ]:
hist_values = (
    token_counts["document_tokens"]
    .hist(bins=[0, 50000, 100000, 150000, 200000, 250000, 300000, 350000, 400000, 450000, 500000])
    .with_columns(pl.col("category").cast(pl.Utf8))
)
hist_values

breakpoint,category,count
f64,str,u32
50000.0,"""[0.0, 50000.0]""",3050
100000.0,"""(50000.0, 100000.0]""",3024
150000.0,"""(100000.0, 150000.0]""",1081
200000.0,"""(150000.0, 200000.0]""",381
250000.0,"""(200000.0, 250000.0]""",156
300000.0,"""(250000.0, 300000.0]""",81
350000.0,"""(300000.0, 350000.0]""",37
400000.0,"""(350000.0, 400000.0]""",7
450000.0,"""(400000.0, 450000.0]""",1


In [24]:
hist_values.plot.bar(y="count", x=alt.X("category", sort=None))

alt.Chart(...)

In [19]:
len([x for x in samples[0]["long_answer_candidates"] if x["top_level"] == True])

14

In [ ]:
samples = df.head().collect().to_pandas().to_dict(orient="index")
samples

In [15]:
len(samples[0]["document_html"])

103968

In [ ]:
tokens = encoding.encode(samples[0]["document_html"])

len(tokens)

34241

In [ ]:
# Alternative simpler approach for mode calculation
from collections import Counter


def get_mode_tokens(annotations):
    """Get mode of start and end tokens from annotations, excluding -1 values"""
    start_tokens = []
    end_tokens = []

    for annotation in annotations:
        if "long_answer" in annotation:
            start_token = annotation["long_answer"].get("start_token", -1)
            end_token = annotation["long_answer"].get("end_token", -1)

            if start_token != -1:
                start_tokens.append(start_token)
            if end_token != -1:
                end_tokens.append(end_token)

    if not start_tokens or not end_tokens:
        return [-1, -1]

    # Get mode (most common value) or None if no valid tokens
    mode_start = Counter(start_tokens).most_common(1)[0][0] if start_tokens else -1
    mode_end = Counter(end_tokens).most_common(1)[0][0] if end_tokens else -1

    return mode_start, mode_end


# Apply to DataFrame using map_elements (for complex operations)
df_with_simple_mode = (
    df.with_columns(
        [
            pl.col("annotations")
            .map_elements(lambda x: get_mode_tokens(x), return_dtype=pl.List(pl.Int64))
            .alias("mode_tokens")
        ]
    )
    .with_columns(
        [
            pl.col("mode_tokens").list.first().alias("mode_start_token"),
            pl.col("mode_tokens").list.last().alias("mode_end_token"),
        ]
    )
    # .with_columns(
    #     [
    #         pl.col("mode_start_token").map_elements(lambda x: print(f"start: {x}, type: {type(x)}") or x),
    #         pl.col("mode_end_token").map_elements(lambda x: print(f"end: {x}, type: {type(x)}") or x),
    #         pl.col("document_tokens").map_elements(lambda x: print(f"len: {len(x)}, {x[:3]}")),
    #     ]
    # )
    .with_columns(
        [
            # Create sliced tokens using mode values
            pl.when((pl.col("mode_start_token") != -1) & (pl.col("mode_end_token") != -1))
            .then(
                pl.col("document_tokens").list.slice(
                    pl.col("mode_start_token"), pl.col("mode_end_token") - pl.col("mode_start_token")
                )
            )
            .otherwise(None)
            .alias("sliced_document_tokens_simple_mode")
        ]
    )
    .with_columns(
        pl.col("sliced_document_tokens_simple_mode")
        .list.eval(pl.element().struct.field("token"))
        .list.join(" ")
        .alias("long_answer_text")
    )
)

print("DataFrame with simple mode calculation:")
df_with_simple_mode.select("document_url", "question_text", "long_answer_text", "document_html").sink_parquet(
    DATA_DIR / "processed/nq_question_answer.parquet"
)

DataFrame with simple mode calculation:


In [55]:
df_with_simple_mode.select("document_url", "question_text", "long_answer_text", "document_html").collect()

KeyboardInterrupt: 

## Create Embeddings

In [1]:
%load_ext autoreload
%autoreload 2
import sys

sys.path.append("/Users/baukebrenninkmeijer/Developer/pydata-2025-context-is-king/generative-benchmarking")

import chromadb
import pandas as pd
import numpy as np
import json
import os
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv
from openai import OpenAI as OpenAIClient

from functions.llm import *
from functions.embed import *
from functions.chroma import *
from functions.evaluate import *
from functions.visualize import *
from rag_pipeline import DATA_DIR
import polars as pl
import polars as pl
from langchain_text_splitters import RecursiveCharacterTextSplitter
from bs4 import BeautifulSoup

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY is not None, "OPENAI_API_KEY is not set"

chroma_client = chromadb.PersistentClient(path=DATA_DIR / "chroma")
openai_client = OpenAIClient(api_key=OPENAI_API_KEY)


2025-08-25 17:24:58.899 | DEBUG    | rag_pipeline:<module>:29 - DATA_DIR=PosixPath('/Users/baukebrenninkmeijer/Developer/pydata-2025-context-is-king/data')


In [2]:
wiki_qa = pl.scan_parquet(DATA_DIR / "processed/nq_question_answer.parquet")
wiki_qa.head().collect()

document_url,question_text,long_answer_text,document_html
str,str,str,str
"""https://en.wikipedia.org//w/in…","""what do the 3 dots mean in mat…","""<P> In logical argument and ma…","""<!DOCTYPE html> <HTML class=""c…"
"""https://en.wikipedia.org//w/in…","""when was the writ watch invent…",null,"""<!DOCTYPE html> <HTML class=""c…"
"""https://en.wikipedia.org//w/in…","""who wrote the song photograph …","""<P> `` Photograph '' is a song…","""<!DOCTYPE html> <HTML class=""c…"
"""https://en.wikipedia.org//w/in…","""who is playing the halftime sh…","""<P> The Super Bowl 50 Halftime…","""<!DOCTYPE html> <HTML class=""c…"
"""https://en.wikipedia.org//w/in…","""star wars the clone wars anaki…","""<P> Matthew MacKendree `` Matt…","""<!DOCTYPE html> <HTML class=""c…"


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=100)
wiki_qa.with_columns(
    pl.col("document_html")
    .map_elements(lambda x: BeautifulSoup(x, "html.parser").get_text(), return_dtype=pl.String)
    .map_elements(lambda x: text_splitter.split_text(x), return_dtype=pl.List(pl.String))
    .alias("chunked_prompt")
).sink_parquet(DATA_DIR / "processed/nq_question_answer_chunked.parquet")

In [3]:
wiki_chunked = pl.scan_parquet(DATA_DIR / "processed/nq_question_answer_chunked.parquet")
wiki_chunked.head(1).collect()

document_url,question_text,long_answer_text,document_html,chunked_prompt
str,str,str,str,list[str]
"""https://en.wikipedia.org//w/in…","""what do the 3 dots mean in mat…","""<P> In logical argument and ma…","""<!DOCTYPE html> <HTML class=""c…","[""Therefore sign - Wikipedia Therefore sign From Wikipedia, the free encyclopedia Jump to: navigation, search ∴ Therefore sign Punctuation apostrophe ’ ' brackets [ ] ( ) { } ⟨ ⟩ colon : comma , ، 、 dash ‒ – — ― ellipsis … ... ⋯ ᠁ ฯ exclamation mark ! full stop, period . guillemets ‹ › « » hyphen ‐"", ""exclamation mark ! full stop, period . guillemets ‹ › « » hyphen ‐ hyphen-minus - question mark ? quotation marks ‘ ’ “ ” ' ' "" "" semicolon ; slash, stroke, solidus / ⁄ Word dividers interpunct · space General typography ampersand & asterisk * at sign @ backslash \ bullet • caret ^ dagger † ‡ degree ° ditto mark ”"", … ""Privacy policy About Wikipedia Disclaimers Contact Wikipedia Developers Cookie statement Mobile view Enable previews""]"


In [4]:
from uuid import uuid4


def generate_ids(n: int) -> list[str]:
    return [str(uuid4()) for _ in range(n)]


documents = (
    pl.scan_parquet(DATA_DIR / "processed/nq_question_answer_chunked.parquet")
    # .limit(50)
    .select("chunked_prompt", "document_url")
    .explode("chunked_prompt")
).collect()
documents = documents.with_columns(pl.Series(generate_ids(documents.height)).alias("unique_id"))
print(documents.shape)
print(documents.select(pl.col("chunked_prompt").str.len_chars().mean()))
documents.head()

(1074844, 3)
shape: (1, 1)
┌────────────────┐
│ chunked_prompt │
│ ---            │
│ f64            │
╞════════════════╡
│ 298.442895     │
└────────────────┘


chunked_prompt,document_url,unique_id
str,str,str
"""Therefore sign - Wikipedia …","""https://en.wikipedia.org//w/in…","""89a43d2f-254a-47dc-9571-9f29a1…"
"""exclamation mark ! full sto…","""https://en.wikipedia.org//w/in…","""b85bf13a-59ca-440b-8e47-96d017…"
"""asterisk * at sign @ backs…","""https://en.wikipedia.org//w/in…","""93159d2d-f1aa-47bb-a7fb-ac1271…"
"""percent, per mil % ‰ plus a…","""https://en.wikipedia.org//w/in…","""c0fe553f-0b51-41eb-9b08-2705a2…"
"""service mark ℠ trademark ™ …","""https://en.wikipedia.org//w/in…","""dede3019-9bd3-422b-8fb3-bf3578…"


In [8]:
corpus_ids = documents.select("unique_id").to_numpy().reshape(-1).tolist()
corpus_documents = documents.select("chunked_prompt").to_numpy().reshape(-1).tolist()
# metadatas = documents.select("document_url").to_dicts()
# print(type(corpus_ids), type(corpus_documents), type(metadatas))

for batch in range(0, len(corpus_documents), 100_000):
    selected_documents = documents.filter(pl.col("unique_id").is_in(corpus_ids[batch : batch + 100_000]))
    corpus_embeddings = openai_embed_in_batches(
        openai_client=openai_client,
        model="text-embedding-3-small",
        texts=selected_documents.select("chunked_prompt").to_numpy().reshape(-1).tolist(),
        batch_size=200,
    )
    selected_documents = selected_documents.with_columns(pl.Series(corpus_embeddings).alias("embedding")).write_parquet(
        DATA_DIR / f"processed/nq_question_answer_chunked_embeddings_{batch}.parquet"
    )

Processing OpenAI batches: 100%|██████████| 375/375 [07:17<00:00,  1.17s/it]


In [2]:
lazy_docs = pl.scan_parquet(DATA_DIR / "processed/nq_question_answer_chunked_embeddings_*.parquet")
lazy_docs.collect_schema()

Schema([('chunked_prompt', String),
        ('document_url', String),
        ('unique_id', String),
        ('embedding', List(Float64))])

In [3]:
# Find strings with invalid characters
import re

pattern = r"[^a-zA-Z0-9._-]"
strings = documents.select("document_url").unique().to_numpy().reshape(-1).tolist()
# Test examples
invalid = []
for s in strings:
    x = extract_title_from_path(s)
    if re.search(pattern, x):
        invalid.append(x)
invalid

NameError: name 'documents' is not defined

In [ ]:
import re

strings = documents.select("document_url").unique().to_numpy().reshape(-1).tolist()


def extract_title_from_path(url):
    pattern = r"title=([^&]+)"
    match = re.search(pattern, url)
    return match.group(1).translate({ord(c): None for c in "!@#$():%,^&!/+=."}) if match else None


for doc in strings:
    print(extract_title_from_path(doc))

1_2B_2_2B_3_2B_4_2B_E28BAF
Lay_Your_Hands_on_Me_Boom_Boom_Satellites_song
High-level_radioactive_waste_management
Sport_utility_vehicle
List_of_Super_Bowl_champions
United_States_declaration_of_war_on_Japan
Spilling_water_for_luck
San_Joaquin_County_California
List_of_Rajya_Sabha_members_from_Assam
An_Officer_and_a_Gentleman
Fender_amplifier
Names_of_the_days_of_the_week
Hard_water
List_of_PokC3A9mon_Advanced_Battle_episodes
Civilian_Conservation_Corps
Kristen_Bell
The_Young_and_the_Restless_cast_members
Sputnik_crisis
Recording_Industry_Association_of_America_certification
Jordan_River
Cast_Away
Forest
IndiGo
I27ll_Name_the_Dogs
Arsenal_FC
Ratatouille_film
The_Pilgrim27s_Progress
Go_Outside_in_the_Rain
Trafalgar_Square_Christmas_tree
Beyond_the_Sea_film
Arms_and_the_Man
List_of_highest-grossing_films
List_of_Miraculous_Tales_of_Ladybug_26_Cat_Noir_episodes
Papa27s_Got_a_Brand_New_Bag
Timothy_McGee
Joseph_Kearns
Grand_slam_baseball
I27m_Just_a_Kid
Life
Gimme_Shelter
History_of_South_Am

In [52]:
for x in [x for x in chroma_client.list_collections() if "wikiqa" in x.name]:
    chroma_client.delete_collection(x.name)

In [7]:
done = []

In [ ]:
collections = [x.name for x in chroma_client.list_collections()]
for file in Path(DATA_DIR / "processed").glob("nq_question_answer_chunked_embeddings_*.parquet"):
    documents = pl.read_parquet(file)
    for id in tqdm(documents.select("document_url").unique().to_numpy().reshape(-1).tolist()):
        collection_name = f"wikiqa_{extract_title_from_path(id)}"
        # logger.info(collection_name, collection_name in collections)
        if collection_name in collections:
            # logger.info(f"Collection {collection_name} already exists")
            continue
        if file.name + collection_name in done:
            continue
        # logger.info(f"Processing {collection_name}")
        corpus_collection = chroma_client.get_or_create_collection(
            name=collection_name, metadata={"hnsw:space": "cosine"}
        )
        selected_docs = documents.filter(pl.col("document_url") == id)
        corpus_ids = selected_docs.select("unique_id").to_numpy().reshape(-1).tolist()
        corpus_documents = selected_docs.select("chunked_prompt").to_numpy().reshape(-1).tolist()
        metadatas = selected_docs.select("document_url").to_dicts()
        embeddings = selected_docs.select("embedding").to_numpy().reshape(-1).tolist()

        # print(len(corpus_ids), len(corpus_documents), len(metadatas), len(selected_docs))

        collection_add_in_batches(
            collection=corpus_collection,
            ids=corpus_ids,
            texts=corpus_documents,
            embeddings=embeddings,
            metadatas=metadatas,
            batch_size=500,
        )
        done.append(file.name + collection_name)


 48%|████▊     | 276/580 [35:00<37:02,  7.31s/it]  